
## Scenario: No Invoice Variation

**Description:** Large fraction of invoices are submitted by a provider with the exact same charge amount.

In [0]:
ENGINE_CATALOG = dbutils.widgets.get("ENGINE_CATALOG")
ENGINE_SCHEMA = dbutils.widgets.get("ENGINE_SCHEMA")

GRAPH_CATALOG = dbutils.widgets.get("GRAPH_CATALOG")
GRAPH_SCHEMA = dbutils.widgets.get("GRAPH_SCHEMA")

In [0]:
%sql

DECLARE execDatetime TIMESTAMP = GETDATE();

In [0]:
df = spark.sql(f"""
  SELECT 
    NOVEL_SCENARIO_ID 
  FROM
    {ENGINE_CATALOG}.{ENGINE_SCHEMA}.T_NOVEL_SCENARIO 
  WHERE 
    NOTEBOOK_NAME = 'Scenario_NoInvoiceVariation'
""")

novelScenarioId = df.collect()[0][0]
print(f"Novel scenario ID: {novelScenarioId}")

## ICPs

In [0]:
# Parameters for ICP invoices
icpInvoiceNumDaysLookback = 365
icpLatestCareNumDaysCutoff = 90
icpMinNumInvoices = 30
icpMinReplicateFraction = 0.95

# Parameters for HHCA invoices
hhcaInvoiceNumDaysLookback = 730
hhcaLatestCareNumDaysCutoff = 90
hhcaMinNumInvoices = 12
hhcaMinReplicateFraction = 0.95

In [0]:
spark.sql(f"""
SELECT * FROM {GRAPH_CATALOG}.gold_knowledge_graph.T_NORM_CLAIM
WHERE CLAIM_STATUS_CODE IN ('Active', 'ASWP', 'Benefit Period', 'Qualification Period')
""").createOrReplaceTempView("active_claims")

In [0]:
spark.sql(f"""
SELECT
  CLAIM_ID,
  CLAIM_NUMBER,
  RES_PERSON_ID,
  INVOICE_SERVICE_START_DATE,
  INVOICE_SERVICE_END_DATE,
  INVOICE_CHARGE_AMT,
  PREV_INVOICE_CHARGE_AMT,
  CASE 
    WHEN INVOICE_CHARGE_AMT = PREV_INVOICE_CHARGE_AMT THEN 1
    ELSE 0
  END AS NO_VARIATION_IND
FROM (
  SELECT
    a.CLAIM_ID,
    c.CLAIM_NUMBER,
    b.RES_PERSON_ID,
    a.INVOICE_SERVICE_START_DATE,
    a.INVOICE_SERVICE_END_DATE,
    a.INVOICE_CHARGE_AMT,
    LAG(a.INVOICE_CHARGE_AMT, 1) OVER (PARTITION BY a.CLAIM_ID, b.RES_PERSON_ID ORDER BY a.INVOICE_SERVICE_START_DATE ASC) AS PREV_INVOICE_CHARGE_AMT
  FROM
    {GRAPH_CATALOG}.{GRAPH_SCHEMA}.T_NORM_INVOICE a
  JOIN
    {GRAPH_CATALOG}.{GRAPH_SCHEMA}.T_RESOLVED_PERSON_INVOICE_CROSSWALK b
    ON a.NORM_INVOICE_ID = b.NORM_INVOICE_ID
    AND b.EDGE_NAME = 'PROVIDED_CARE_ON_INVOICE'
  JOIN
    active_claims c
    ON a.CLAIM_ID = c.CLAIM_ID
  WHERE
    a.INVOICE_SERVICE_END_DATE >= DATE_SUB(GETDATE(), {icpInvoiceNumDaysLookback})
)
""").createOrReplaceTempView("icp_invoice")

In [0]:
spark.sql(f"""
SELECT
  a.CLAIM_ID,
  a.CLAIM_NUMBER,
  a.RES_PERSON_ID AS PROVIDER_ID,
  CONCAT(b.FIRST_NAME, ' ', b.LAST_NAME) AS PROVIDER_NAME,
  'ICP' AS PROVIDER_TYPE,
  a.NUM_INVOICES,
  a.NUM_REPLICATE_INVOICES,
  TRY_DIVIDE(a.NUM_REPLICATE_INVOICES, a.NUM_INVOICES) AS FRACTION_REPLICATE_INVOICES,
  a.CARE_END_DATE
FROM (
  SELECT
    CLAIM_ID,
    CLAIM_NUMBER,
    RES_PERSON_ID,
    COUNT(*) AS NUM_INVOICES,
    SUM(NO_VARIATION_IND) AS NUM_REPLICATE_INVOICES,
    MAX(INVOICE_SERVICE_END_DATE) AS CARE_END_DATE
  FROM
    icp_invoice
  GROUP BY
    CLAIM_ID,
    CLAIM_NUMBER,
    RES_PERSON_ID
) a
JOIN
  {GRAPH_CATALOG}.{GRAPH_SCHEMA}.T_RESOLVED_PERSON b
  ON a.RES_PERSON_ID = b.RES_PERSON_ID
WHERE
  CARE_END_DATE >= date_sub(getdate(), {icpLatestCareNumDaysCutoff})
""").createOrReplaceTempView("icp_invoice_trans_1")

In [0]:
spark.sql(f"""
SELECT 
  * 
FROM 
  icp_invoice_trans_1 
WHERE 
  FRACTION_REPLICATE_INVOICES >= {icpMinReplicateFraction} 
  AND NUM_INVOICES >= {icpMinNumInvoices}         
""").createOrReplaceTempView("icp_flagged_claims")

## HHCAs

In [0]:
spark.sql(f"""
SELECT
  CLAIM_ID,
  CLAIM_NUMBER,
  RES_BUSINESS_ID,
  CARE_TYPE,
  INVOICE_SERVICE_START_DATE,
  INVOICE_SERVICE_END_DATE,
  INVOICE_CHARGE_AMT,
  PREV_INVOICE_CHARGE_AMT,
  CASE 
    WHEN INVOICE_CHARGE_AMT = PREV_INVOICE_CHARGE_AMT THEN 1
    ELSE 0
  END AS NO_VARIATION_IND
FROM (
  SELECT
    a.CLAIM_ID,
    d.CLAIM_NUMBER,
    b.RES_BUSINESS_ID,
    c.BUSINESS_TYPE AS CARE_TYPE,
    a.INVOICE_SERVICE_START_DATE,
    a.INVOICE_SERVICE_END_DATE,
    a.INVOICE_CHARGE_AMT,
    LAG(a.INVOICE_CHARGE_AMT, 1) OVER (PARTITION BY a.CLAIM_ID, b.RES_BUSINESS_ID ORDER BY a.INVOICE_SERVICE_START_DATE ASC) AS PREV_INVOICE_CHARGE_AMT
  FROM
    {GRAPH_CATALOG}.{GRAPH_SCHEMA}.T_NORM_INVOICE a
  JOIN
    {GRAPH_CATALOG}.{GRAPH_SCHEMA}.T_RESOLVED_BUSINESS_INVOICE_CROSSWALK b
    ON a.NORM_INVOICE_ID = b.NORM_INVOICE_ID
    AND b.EDGE_NAME = 'PROVIDED_CARE_ON_INVOICE'
  JOIN
    {GRAPH_CATALOG}.{GRAPH_SCHEMA}.T_RESOLVED_BUSINESS c
    ON b.RES_BUSINESS_ID = c.RES_BUSINESS_ID
  JOIN
    active_claims d
    ON a.CLAIM_ID = d.CLAIM_ID
  WHERE
    a.INVOICE_SERVICE_END_DATE >= DATE_SUB(GETDATE(), {hhcaInvoiceNumDaysLookback})
    AND c.BUSINESS_TYPE = 'HHCA'
)
""").createOrReplaceTempView("hhca_invoice")

In [0]:
spark.sql(f"""
SELECT
  a.CLAIM_ID,
  a.CLAIM_NUMBER,
  a.RES_BUSINESS_ID AS PROVIDER_ID,
  b.BUSINESS_NAME AS PROVIDER_NAME,
  a.CARE_TYPE AS PROVIDER_TYPE,
  a.NUM_INVOICES,
  a.NUM_REPLICATE_INVOICES,
  TRY_DIVIDE(a.NUM_REPLICATE_INVOICES, a.NUM_INVOICES) AS FRACTION_REPLICATE_INVOICES,
  a.CARE_END_DATE
FROM (
  SELECT
    CLAIM_ID,
    CLAIM_NUMBER,
    RES_BUSINESS_ID,
    CARE_TYPE,
    COUNT(*) AS NUM_INVOICES,
    SUM(NO_VARIATION_IND) AS NUM_REPLICATE_INVOICES,
    MAX(INVOICE_SERVICE_END_DATE) AS CARE_END_DATE
  FROM
    hhca_invoice
  GROUP BY
    CLAIM_ID,
    CLAIM_NUMBER,
    RES_BUSINESS_ID,
    CARE_TYPE
) a
JOIN
  {GRAPH_CATALOG}.{GRAPH_SCHEMA}.T_RESOLVED_BUSINESS b
  ON a.RES_BUSINESS_ID = b.RES_BUSINESS_ID
WHERE
  CARE_END_DATE >= DATE_SUB(GETDATE(), {hhcaLatestCareNumDaysCutoff})
""").createOrReplaceTempView("hhca_invoice_trans_1")

In [0]:
spark.sql(f"""
SELECT 
  * 
FROM 
  hhca_invoice_trans_1 
WHERE 
  FRACTION_REPLICATE_INVOICES >= {hhcaMinReplicateFraction} 
  AND NUM_INVOICES >= {hhcaMinNumInvoices}         
""").createOrReplaceTempView("hhca_flagged_claims")

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW flagged_claims AS
SELECT * FROM icp_flagged_claims
UNION ALL
SELECT * FROM hhca_flagged_claims;

In [0]:
spark.sql(f"""
    INSERT INTO {ENGINE_CATALOG}.{ENGINE_SCHEMA}.T_SCENARIO_NO_INVOICE_VARIATION_DETAIL
    SELECT 
      *,
      GETDATE() AS FEATURE_DATETIME
    FROM flagged_claims;
""")